<center>

### School of Digital Technologies  
#### ITM-390: Machine Learning
#### Lab Practices: Ensemble Models for Classification

**Lecturer:** Kuntha Pin (Ph.D Eng.)

</center>

---

### Contents
### 1. Introduction
### 2. Various Techniques in Ensemble Learning
---

## 1. Introduction

In the previous session, ensemble learning concepts were introduced, including hard voting, soft voting, bagging, boosting, and stacking. These methods aim to improve model performance by combining multiple models.

**Why Ensemble Methods?**

Ensemble methods work because combining diverse models often produces better predictions than any single model. Key reasons:
- **Reduces Bias & Variance**: Different models capture different patterns; combining them reduces prediction error
- **Improves Generalization**: Less likely to overfit when models disagree on borderline cases
- **Robustness**: If one model fails on certain instances, others may succeed

In this lab, a Jupyter Notebook is provided that integrates both code and explanations. The goal is to help students understand how ensemble methods work in practice through hands-on experiments.

**Dataset: Digits**

Throughout this lab, we use sklearn's Digits dataset. It contains 1,797 grayscale images of handwritten digits (0-9), each represented as an 8x8 pixel grid flattened into 64 features. With 10 classes and a reasonable sample size, it gives ensemble methods enough room to show meaningful differences in performance.

**Core Concepts**
- **Bias**: Systematic error from model oversimplification
- **Variance**: Error from model sensitivity to training data variations
- **Diversity**: Different models should make different predictions (key to ensemble success)
- **Aggregation**: How predictions from multiple models are combined

**Learning Objectives**
- Implement and compare different ensemble methods
- Analyze the impact of ensemble techniques on accuracy and variance
- Explain when each ensemble method is most effective


## 2. Various Techniques in Ensemble Learning

### 2.1 Simple Ensemble Techniques: Voting Classifier

#### Understanding Voting Classifiers

The **max voting method** (also called majority voting) is a fundamental ensemble technique for classification. The core idea:
1. Train multiple diverse classifiers on the same data
2. For each new sample, collect predictions from all classifiers
3. Use the majority vote (hard voting) or average probability (soft voting) for the final prediction

**Why does voting work?**
- Each base classifier makes different errors. By voting, uncorrelated errors tend to cancel out
- Averaging reduces noise while preserving signal
- Simple but effective with no complex dependencies between base models

**Hard vs. Soft Voting**
- **Hard Voting**: Each model casts one vote. Final prediction = most frequent class. Works with any classifier, even those without probability estimates.
- **Soft Voting**: Each model outputs class probabilities. Final prediction = class with highest average probability. More nuanced, but requires all classifiers to support probability output.

Now let us use the Digit dataset to demonstrate the Voting Classifier.


### - Loading Digits Dataset


In [12]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data   # 64 features (8x8 pixel values)
Y = digits.target  # 10 classes (digits 0-9)

print("Features shape:", X.shape)
print("Target shape:", Y.shape)
print("Classes:", list(digits.target_names))


Features shape: (1797, 64)
Target shape: (1797,)
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [13]:
# Split data into train and test set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,Y,
                                                    test_size = 0.20,
                                                    random_state = 42)

print("X_train.shape, y_train.shape: ", X_train.shape, y_train.shape)
print("X_test.shape, y_test.shape: ", X_test.shape, y_test.shape)

X_train.shape, y_train.shape:  (1437, 64) (1437,)
X_test.shape, y_test.shape:  (360, 64) (360,)


In [14]:
# Ensemble of Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.ensemble import VotingClassifier

estimators = []
estimators.append(('LR', LogisticRegression(solver='lbfgs', max_iter=1000)))
estimators.append(('SVC', SVC(gamma='auto', probability=True)))
estimators.append(('DTC', DecisionTreeClassifier(random_state=42)))


#### Voting Classifier supports two types of votings.

#### Hard Voting: Majority Vote Rule

In hard voting, each classifier casts one vote for their predicted class, and the class with the most votes wins.

**Example**: If three classifiers predict (Class A, Class A, Class B), the ensemble predicts Class A (2 votes vs 1 vote).

**Mathematical Representation**:
- For sample x: Each classifier $h_i$ outputs class $\hat{y}_i$
- Final prediction: $\hat{y}_{ensemble} = \text{mode}(\hat{y}_1, \hat{y}_2, ..., \hat{y}_n)$

Hard voting is simple and fast. It works with any classifier since no probability calibration is needed. The downside is that all classifiers are weighted equally, so it ignores how confident each model is.


In [15]:
# 1. Voting Classifier with hard voting
hard_voting = VotingClassifier(estimators=estimators, voting='hard')
hard_voting.fit(X_train, y_train)
y_pred = hard_voting.predict(X_test)


# accuracy_score metric to predict Accuracy
from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

c:\Users\U-ser\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Accuracy: 0.9111111111111111


#### Soft Voting: Probability-Weighted Averaging

In soft voting, classifiers output probability predictions for each class, and we average these probabilities across all classifiers. The class with the highest average probability wins.

**Example**:
- Classifier 1 predicts: P(A)=0.70, P(B)=0.30
- Classifier 2 predicts: P(A)=0.60, P(B)=0.40  
- Classifier 3 predicts: P(A)=0.80, P(B)=0.20
- **Ensemble**: Avg P(A)=0.70, Avg P(B)=0.30 -> Predicts **Class A**

Compare with hard voting: (A, A, A) -> Predicts A. Same result here, but soft voting provides more nuance.

**Mathematical Representation**:
- For sample x, probability of class c from classifier i: $p_{i,c}$
- Final prediction: $\hat{y}_{ensemble} = \arg\max_c \frac{1}{n} \sum_{i=1}^{n} p_{i,c}$

Soft voting generally performs better than hard voting because it considers model confidence. The tradeoff is that it requires probability estimates from all classifiers, and is sensitive to poorly calibrated probabilities.


In [16]:
# Voting Classifier with soft voting
soft_voting = VotingClassifier(estimators=estimators, voting='soft')
soft_voting.fit(X_train, y_train)
y_pred = soft_voting.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

c:\Users\U-ser\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Accuracy: 0.9277777777777778


**Analysis of Results**:
- Soft voting often outperforms hard voting because it uses probability estimates
- On simple datasets like Iris with small test sets, the difference may not be obvious
- The Iris dataset is very small (150 samples total), so results can vary

**Think about this**: If hard and soft voting give very similar results, what does that tell you about the diversity of your base classifiers? Does dataset size play a role?

---


### 2.2 Advanced Ensemble Techniques

#### 2.2.1 Bagging (Bootstrap Aggregating)

**What is Bagging?**

Bagging combines two ideas:
- **Bootstrapping**: Randomly resampling training data WITH replacement to create multiple subsets
- **Aggregating**: Training independent models on each subset and combining their predictions

**How Bagging Reduces Variance**

Imagine you have 100 training samples. Bagging works like this:
1. Random sample WITH replacement -> Dataset 1 -> Train Model 1
2. Random sample WITH replacement -> Dataset 2 -> Train Model 2
3. ... repeat this 10-100 times ...
4. **Aggregate** predictions: For classification, use majority voting. For regression, use average.

If individual models have variance $\sigma^2$ and their errors are perfectly uncorrelated, ensemble variance drops to $\approx \frac{\sigma^2}{n}$. In practice, bagged models are not fully independent (they share the same algorithm and overlapping data), so the reduction is smaller than $\frac{1}{n}$, but still significant.

**Key Trade-off**

- **Larger subsamples** -> Higher individual model accuracy BUT less diversity
- **Smaller subsamples** -> Lower individual accuracy BUT more diversity
- **Sweet spot**: Typically 60-100% of training data

**When Bagging Works Best:**
- High variance models like Decision Trees (very sensitive to training data)
- Limited training data where resampling maximizes data usage
- Need to reduce overfitting

**When Bagging Does Not Help Much:**
- Stable learners like KNN or Linear Regression (already low variance)
- Bias-dominated problems (bagging reduces variance, not bias)

**Popular Bagging Variants:**
- **Random Forest**: Bagging of Decision Trees + random feature selection
- **Extra Trees**: Similar to RF but with more randomness in split thresholds



In [17]:
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import BaggingClassifier

# Model evaluation
from sklearn.model_selection import cross_val_score

# Using the same Digits dataset
digits = load_digits()
X, y = digits.data, digits.target

# Create classifiers
rf = RandomForestClassifier(random_state=42)
et = ExtraTreesClassifier(random_state=42)
knn = KNeighborsClassifier()
svc = SVC()
ridge = RidgeClassifier()

classifiers = [rf, et, knn, svc, ridge]
seed = 42

for clf in classifiers:
    vanilla_scores = cross_val_score(clf, X, y, cv=10, n_jobs=-1)
    bagging_clf = BaggingClassifier(clf, max_samples=0.4, max_features=0.75, random_state=seed)
    bagging_scores = cross_val_score(bagging_clf, X, y, cv=10, n_jobs=-1)

    print("Mean of: {1:.3f}, std: (+/-) {2:.3f} [{0}]".format(clf.__class__.__name__, vanilla_scores.mean(), vanilla_scores.std()))
    print("Mean of: {1:.3f}, std: (+/-) {2:.3f} [Bagging {0}]\n".format(clf.__class__.__name__, bagging_scores.mean(), bagging_scores.std()))


Mean of: 0.949, std: (+/-) 0.023 [RandomForestClassifier]
Mean of: 0.945, std: (+/-) 0.028 [Bagging RandomForestClassifier]

Mean of: 0.965, std: (+/-) 0.021 [ExtraTreesClassifier]
Mean of: 0.956, std: (+/-) 0.023 [Bagging ExtraTreesClassifier]

Mean of: 0.971, std: (+/-) 0.017 [KNeighborsClassifier]
Mean of: 0.964, std: (+/-) 0.018 [Bagging KNeighborsClassifier]

Mean of: 0.970, std: (+/-) 0.021 [SVC]
Mean of: 0.964, std: (+/-) 0.021 [Bagging SVC]

Mean of: 0.905, std: (+/-) 0.038 [RidgeClassifier]
Mean of: 0.907, std: (+/-) 0.036 [Bagging RidgeClassifier]



**Analysis of Bagging Results:**


**What We Should Observe:**
1. **Lower std**: Variance reduced by averaging multiple models
2. **Higher mean accuracy**: Ensemble typically outperforms individual models
3. **Exception (SVC)**: SVC is already quite stable due to regularization. Bagging helps unstable learners more.

**Key Takeaway**: Bagging is most effective on decision trees and KNN (unstable, high-variance learners). Stable learners like linear models show minimal improvement.

---

**So our bagged individual classifiers are (mostly) better, but which one should we choose?**

Instead of selecting one, let's combine them using a **Voting Classifier** to see how they perform together.


In [18]:
from sklearn.ensemble import VotingClassifier

ensemble_clf = VotingClassifier(
    estimators=[
        ('Random Forests', rf),
        ('Extra Trees', et),
        ('KNeighbors', knn),
        ('SVC', svc),
        ('Ridge Classifier', ridge)],
    voting='hard')

for model, label in zip([rf, et, knn, svc, ridge, ensemble_clf],
                        ['Random Forest', 'Extra Trees', 'KNeighbors', 'SVC', 'Ridge Classifier', 'Ensemble']):
    scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')
    print("Accuracy: %0.2f (+/- %0.2f) [%s]" % (scores.mean(), scores.std(), label))

Accuracy: 0.95 (+/- 0.02) [Random Forest]
Accuracy: 0.96 (+/- 0.02) [Extra Trees]
Accuracy: 0.97 (+/- 0.02) [KNeighbors]
Accuracy: 0.97 (+/- 0.02) [SVC]
Accuracy: 0.91 (+/- 0.04) [Ridge Classifier]
Accuracy: 0.97 (+/- 0.02) [Ensemble]


With our ensemble results shown above, we can see how the combined Voting Classifier compares against each individual model.

Notice that the ensemble accuracy is competitive with or better than most individual models, while also showing reduced variance. This is the core benefit of ensembling: more stable, reliable predictions.


#### 2.2.2 Boosting: Learning from Mistakes

**Core Idea of Boosting:**

Unlike Bagging (which trains models independently), Boosting trains models **sequentially**. Each new model learns from the errors of previous models.

**How Boosting Works:**
1. Train Model 1 on original data
2. **Identify mistakes**: Which samples did Model 1 misclassify?
3. Train Model 2 with **higher weight on misclassified samples**
4. **Identify new mistakes**: Find samples Model 2 still gets wrong
5. Train Model 3 with **even higher weight** on difficult samples
6. **Combine all models**: Weighted average (better-performing models get higher weight)

**Mathematical Intuition:**

Boosting is like gradient descent at the ensemble level:
- Model 1 makes predictions y1
- Error is e1 = y - y1
- Model 2 tries to predict e1 (correct the mistakes)
- Final prediction: y_ensemble = y1 + learning_rate * y2 + ...

**Why Boosting Works:**
- **Reduces Bias**: Sequential models learn to fix systematic errors
- **Focuses on hard examples**: Gradually improves on difficult samples
- Often beats bagging in practice

**Limitations:**
- Sequential training is slower than parallel bagging
- Sensitive to outliers (can assign very high weights to noisy samples)
- Risk of overfitting if too many iterations
- Parameters need tuning (learning rate, number of estimators)

**Popular Boosting Variants:**


In [19]:
from sklearn.datasets import load_digits

# Continuing with the Digits dataset for the boosting section
X, y = load_digits().data, load_digits().target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)


**- AdaBoost (Adaptive Boosting):**

One of the simplest and earliest boosting algorithms:
1. Start with equal weights for all samples
2. Train a shallow decision tree (often depth=1, called a "stump")
3. Calculate error: how many samples did this tree misclassify?
4. Increase weights for misclassified samples
5. Train next tree with adjusted weights
6. Final prediction: weighted majority vote (accurate trees get higher weight)

AdaBoost is simple and works well even with very weak learners. The main downside is that it is sensitive to outliers, since misclassified samples keep getting higher weights.


In [20]:
from sklearn.ensemble import AdaBoostClassifier
ada_boost = AdaBoostClassifier(random_state=1)
ada_boost.fit(X_train, y_train)
ada_boost.score(X_test,y_test)

0.7527777777777778

**- Gradient Boosting:**

Gradient Boosting improves on AdaBoost by using a different strategy to correct mistakes. Instead of reweighting samples, it fits each new tree on the **residual errors** (the gap between the true values and the current predictions).

**How it differs from AdaBoost:**
- AdaBoost adjusts sample weights so the next tree focuses on hard examples
- Gradient Boosting computes the gradient of the loss function and trains the next tree to predict that gradient (i.e., the direction to reduce the error)

**Key parameters:**
- `learning_rate`: Shrinks each tree's contribution. Smaller values need more trees but generalize better
- `n_estimators`: Number of sequential trees to build
- `max_depth`: Depth of each individual tree (usually kept shallow, e.g., 3-5)

Gradient Boosting is slower than AdaBoost because it builds deeper trees and computes gradients. It is available in sklearn as `GradientBoostingClassifier`.


In [21]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(random_state=1)
gb_clf.fit(X_train, y_train)
print("Gradient Boosting Accuracy:", gb_clf.score(X_test, y_test))


Gradient Boosting Accuracy: 0.9583333333333334


**- XGBoost (eXtreme Gradient Boosting):**

An advanced gradient boosting implementation with major improvements over basic implementations like sklearn's `GradientBoostingClassifier`:

**Key Features:**
- **Gradient-based learning**: Uses gradients of the loss function (more principled than AdaBoost's sample reweighting)
- **Regularization built-in**: L1 and L2 regularization reduce overfitting
- **Tree pruning**: Can simplify complex trees automatically
- **Handles sparse data**: Can work with missing values efficiently

XGBoost has a highly optimized C++ backend, making it significantly faster than sklearn's gradient boosting. It consistently wins machine learning competitions and generally has better default parameters than other boosting implementations.

The main tradeoff is complexity: XGBoost has more hyperparameters to tune compared to AdaBoost.


In [ ]:
import sys
!{sys.executable} -m pip install xgboost

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 800.7 kB/s eta 0:01:01
   ---------------------------------------- 0.5/48.9 MB 800.7 kB/s eta 0:01:01
   ---------------------------------------- 0.5/48.9 MB 800.7 kB/s eta 0:01:01
    --------------------------------------- 0.8/48.9 MB 509.7 kB/s eta 0:01:35
    --------------------------------------- 0.8/48.9 MB 509.7 kB/s eta 0:01:35
    --------------------------------------- 0.8/48.9 MB 509.7 kB/s eta 0:01:35
    --------------------------------------- 0.8/48.9 MB 509.7 kB/s eta 0:01:35
    --------------------------------------- 0.8/48.9 MB 509.7 kB/s eta 0:01:35
    --------------

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 1.8 MB/s eta 0:00:27
    --------------------------------------- 0.8/48.9 MB 1.8 MB/s eta 0:00:27
    --------------------------------------- 1.0/48.9 MB 1.3 MB/s eta 0:00:38
   - -------------------------------------- 1.3/48.9 MB 1.2 MB/s eta 0:00:39
   - -------------------------------------- 1.6/48.9 MB 1.2 MB/s eta 0:00:39
   - -------------------------------------- 1.8/48.9 MB 1.3 MB/s eta 0:00:35
   - -------------------------------------- 2.1/48.9 MB 1.3 MB/s eta 0:00:38
   - -------------------------------------- 2.4/48.9 MB 1.2 MB/s eta 0:00:39
   -- ----------------------------

In [23]:
from xgboost import XGBClassifier

xgb_boost = XGBClassifier(
    random_state=1,
    learning_rate=0.01
)

xgb_boost.fit(X_train, y_train)
accuracy = xgb_boost.score(X_test, y_test)

print("Accuracy:", accuracy)


Accuracy: 0.9416666666666667


In [24]:
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score

# Compare all three boosting methods and their ensemble
ensemble_clf = VotingClassifier(
    estimators=[
        ('AdaBoost', ada_boost),
        ('GradientBoosting', gb_clf),
        ('XGBoost', xgb_boost)],
    voting='hard'
)

models = [ada_boost, gb_clf, xgb_boost, ensemble_clf]
labels = ['AdaBoost', 'GradientBoosting', 'XGBoost', 'Ensemble']

for model, label in zip(models, labels):
    scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')
    print("Accuracy: %0.2f (+/- %0.2f) [%s]" % (scores.mean(), scores.std(), label))


Accuracy: 0.71 (+/- 0.06) [AdaBoost]
Accuracy: 0.93 (+/- 0.03) [GradientBoosting]
Accuracy: 0.89 (+/- 0.04) [XGBoost]
Accuracy: 0.91 (+/- 0.04) [Ensemble]


#### 2.2.3 Stacking: Meta-Learning Ensemble

**Core Concept of Stacking:**

Stacking (also called **Stacked Generalization**) uses predictions from multiple base models as features to train a meta-model.

**Two-Level Architecture:**

**Level 0 (Base Layer)**: Multiple diverse classifiers (e.g., Decision Tree, KNN, SVM) that each capture different patterns in the data.

**Level 1 (Meta Layer)**: A meta-learner (usually something simple like Logistic Regression) that takes Level 0 predictions as input and learns how to best combine them.

**How Stacking Works:**

**Training Phase:**
1. Divide training data into k folds
2. For each fold: train Level 0 models on the remaining data, generate predictions on the held-out fold
3. Stack these predictions as a new feature set
4. Train the Level 1 model on these meta-features

**Testing Phase:**
1. Level 0 models predict on test data
2. These predictions become features for the Level 1 model
3. Level 1 model produces the final prediction

The meta-learner essentially learns:
- Which base models are trustworthy?
- For which types of examples should we trust Model A vs Model B?
- How to optimally combine conflicting predictions?

**Example Intuition:**
- Decision Tree says: "Class A" (90% confidence)
- KNN says: "Class B" (70% confidence)
- SVM says: "Class A" (60% confidence)

Meta-learner learns: "When Tree and SVM agree despite different confidence levels, trust them 80% of the time"

**Advantages of Stacking:**
- Most flexible: can combine any models
- Learns optimal combination strategy
- Often best performance on difficult problems
- Can use heterogeneous base models (different algorithms)

**Disadvantages of Stacking:**
- More complex (more hyperparameters to tune)
- Computationally expensive (train k+1 models)
- Risk of overfitting if not careful with fold strategy
- Requires sufficient training data
- Harder to interpret than simpler ensembles

**When to Use Stacking:**
- Combining very diverse models (different algorithm types)
- Have sufficient training data
- Accuracy is paramount (willing to trade complexity)
- Other ensembles not giving good results


In [25]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Base models (level-0 models)
estimators = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('knn', KNeighborsClassifier()),
    ('svm', SVC(probability=True))
]

# Meta-model (level-1 model)
meta_model = LogisticRegression(max_iter=1000) # classification

# Stacking classifier
stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    cv=5  # cross-validation for training meta-model
)

# Train
stack_clf.fit(X_train, y_train)

# Predict
y_pred = stack_clf.predict(X_test)

# Evaluate
from sklearn.metrics import accuracy_score
print("Stacking Accuracy:", accuracy_score(y_test, y_pred))


c:\Users\U-ser\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\U-ser\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\U-ser\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\U-ser\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\

Stacking Accuracy: 0.9944444444444445


---

## 3. Student Tasks & Comprehension Questions

### 3.1 Comprehension Questions

**Q1. Voting Classifier Basics**
1. Why might soft voting outperform hard voting?
2. Can hard voting work with classifiers that don't output probabilities? (Y/N + explain)

**Q2. Bagging Fundamentals**
1. Why does bagging reduce variance but typically not bias?
2. If you have a dataset with 1000 samples, and you create bootstrap samples with `max_samples=0.4`, how many unique samples do you expect in each bootstrap sample? (Hint: Mathematical formula of bootstrap)

**Q3. Boosting Concepts**
1. How does boosting differ from bagging in terms of sample selection?
2. In AdaBoost, what happens to sample weights after each iteration?
3. Why might boosting be more susceptible to outliers than bagging?
4. What does "regularized" in XGBoost mean, and why is it important?
5. How does Gradient Boosting differ from AdaBoost in how it corrects errors?

**Q4. Stacking Logic**
1. What is the meta-model in stacking?
2. Name one advantage and one disadvantage of stacking over voting.

---


## Answer

**Q1**

1. Soft voting averages the predicted probabilities of all classifiers, so it takes into account how confident each model is. Hard voting gives every classifier exactly one vote, so it ignores confidence. A very confident correct model can therefore outweigh weak, uncertain ones in soft voting.
2. Yes. Hard voting only needs each classifier's predicted class, and the class with the most votes wins. No probability estimates or calibration are needed, so it works with any classifier. Soft voting is the one that requires probabilities.

**Q2**

1. Bagging trains many models on different bootstrap samples and averages (or votes) their predictions. Averaging cancels out the random errors caused by the training data, so variance goes down (about σ²/n if the models were fully independent). But every model in the ensemble is the same type and makes the same kind of systematic error, so averaging them does not remove that error, and bias stays about the same.
2. 
- A bootstrap sample draws 0.4 × 1000 = 400 samples with replacement.
- The chance that one particular sample is NOT picked in one draw is 1 − 1/1000 = 0.999.
- The chance that it is not picked in any of the 400 draws is 0.999^400 ≈ 0.670.
- So the chance that it IS picked at least once is 1 − 0.670 = 0.330.
- Expected unique samples = 1000 × 0.330 ≈ **330 unique samples** (the other 70 draws are repeats of the same samples).

**Q3**

1. Bagging creates each model's training set by randomly resampling the data with replacement, and the models are trained independently (in parallel), so every sample has the same chance of being picked. Boosting trains models one after another (sequentially), and each new model pays more attention to the samples the previous models got wrong, so the selection or weighting is not random but focused on the hard examples.

2. All samples start with equal weights. After each iteration, the weights of the misclassified samples are increased (and the weights of correctly classified samples become relatively smaller), so the next tree focuses on the difficult samples.

3. Outliers or noisy samples are often misclassified, so boosting keeps increasing their weights in every iteration, and later models are forced to concentrate on them, which can hurt the final model. In bagging, each model is trained independently and an outlier appears in only some of the bootstrap samples, and the final vote averages its effect out.

4. It means XGBoost has L1 and L2 regularization built in, which adds a penalty on complex trees (for example, very large leaf weights). This keeps the model simpler and reduces overfitting, so it generalizes better to new data.

5. AdaBoost corrects errors by changing the sample weights so the next tree focuses on the misclassified samples. Gradient Boosting does not reweight samples. It fits each new tree on the residual errors (the gradient of the loss function), so each new tree predicts the direction that reduces the current error.

**Q4**

1. The meta-model (the Level 1 model, usually something simple like Logistic Regression) is trained on the predictions of the Level 0 base models. It takes these predictions as its input features and learns how to best combine them into the final prediction.

2. 
- Advantage: stacking learns the best way to combine the models (which models to trust and when), while voting treats all models equally.
- Disadvantage: stacking is more complex and more expensive to train (it needs cross-validation and an extra model), and it has a higher risk of overfitting if the folds are not handled carefully.



### 3.2 Application & Analysis Tasks (Application Level)

**Choose one of the following tasks to complete.**

**Task 1: Ensemble Comparison Experiment**
```
Objective: Compare all ensemble methods on a new dataset
Instructions:
1. Load any dataset from sklearn (Wine, Breast Cancer, or Iris)
2. Implement all ensemble methods:
   - Voting (both hard and soft)
   - Bagging (with RandomForest)
   - Boosting (AdaBoost, GradientBoosting, and XGBoost)
   - Stacking (with 3 diverse base models)
3. Create a comparison table showing:
   - Accuracy on test set
   - Cross-validation mean & std
4. Visualize: Bar plot comparing accuracies
5. Analyze: Which ensemble method performed best? Why?
```

**Task 2: Understand Diversity's Impact**
```
Objective: Investigate how base model diversity affects ensemble performance
Instructions:
1. Create a voting classifier with:
   - Experiment A: 3 identical Decision Trees (low diversity)
   - Experiment B: 3 different algorithms (LogisticRegression, KNN, SVM)
2. Compare accuracies and variances
3. Why is Experiment B expected to perform better?
4. What would happen if all base models achieved 99% accuracy individually?
   Can the ensemble beat this?
```

---


 <center>

# End of Class
+++++++++++++++++++++++++++++++++++

Good luck with your studies! Remember:
✓ Ensembles are powerful, but understand WHY they work
✓ Diversity in base models is crucial
✓ More is not always better -- diminishing returns exist
✓ Match your ensemble method to your problem

:)
---
</center>


In [26]:
# Imports we need for Task 2
from sklearn.datasets import load_digits
from sklearn.ensemble import VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Load the Digits dataset again (X = pixel features, y = the digit 0-9)
X, y = load_digits(return_X_y=True)

# Check that it loaded
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1797, 64)
y shape: (1797,)


In [27]:
# EXPERIMENT A: 3 IDENTICAL Decision Trees (low diversity)
# All three trees use random_state=42, so they are built the exact same way
# and will always make the same predictions.
exp_a = VotingClassifier(
    estimators=[
        ('tree1', DecisionTreeClassifier(random_state=42)),
        ('tree2', DecisionTreeClassifier(random_state=42)),
        ('tree3', DecisionTreeClassifier(random_state=42)),
    ],
    voting='hard'   # hard voting = majority vote
)

# Test it with 10-fold cross-validation:
# the data is split into 10 parts, and the model is tested 10 times,
# each time on a different part, which gives 10 accuracy scores
scores_a = cross_val_score(exp_a, X, y, cv=10, scoring='accuracy')

# Print the average accuracy and the std (how much the 10 scores differ)
print("Experiment A (3 identical trees)")
print("   Mean accuracy:", round(scores_a.mean(), 3))
print("   Std:", round(scores_a.std(), 3))

Experiment A (3 identical trees)
   Mean accuracy: 0.834
   Std: 0.038


In [28]:
# EXPERIMENT B: 3 DIFFERENT algorithms (high diversity)
# Each one learns in a different way:
#   LogisticRegression = a linear model
#   KNeighborsClassifier = looks at the nearest training samples
#   SVC = a support vector machine (draws a curved boundary between classes)
exp_b = VotingClassifier(
    estimators=[
        ('LR', LogisticRegression(max_iter=1000)),
        ('KNN', KNeighborsClassifier()),
        ('SVM', SVC()),
    ],
    voting='hard'   # hard voting = majority vote
)

# Test it with the same 10-fold cross-validation as Experiment A
scores_b = cross_val_score(exp_b, X, y, cv=10, scoring='accuracy')

# Print the average accuracy and the std
print("Experiment B (LR + KNN + SVM)")
print("   Mean accuracy:", round(scores_b.mean(), 3))
print("   Std:", round(scores_b.std(), 3))

Experiment B (LR + KNN + SVM)
   Mean accuracy: 0.969
   Std: 0.02


In [29]:
# A SINGLE Decision Tree, to compare against the ensembles
single_tree = DecisionTreeClassifier(random_state=42)
scores_single = cross_val_score(single_tree, X, y, cv=10, scoring='accuracy')

# Print a small summary table of all three results
print("Model                              Mean     Std")
print("Single Decision Tree              ", round(scores_single.mean(), 3), "  ", round(scores_single.std(), 3))
print("Experiment A (3 identical trees)  ", round(scores_a.mean(), 3), "  ", round(scores_a.std(), 3))
print("Experiment B (LR + KNN + SVM)     ", round(scores_b.mean(), 3), "  ", round(scores_b.std(), 3))

Model                              Mean     Std
Single Decision Tree               0.834    0.038
Experiment A (3 identical trees)   0.834    0.038
Experiment B (LR + KNN + SVM)      0.969    0.02


## Task 2: Understand Diversity's Impact

**2. Compare accuracies and variances**

| Model | Mean accuracy | Std |
|---|---|---|
| Single Decision Tree | 0.834 | 0.038 |
| Experiment A (3 identical trees) | 0.834 | 0.038 |
| Experiment B (LR + KNN + SVM) | 0.969 | 0.02 |

Experiment A has the same accuracy and the same std as the single tree, which means that voting with three identical trees gave no benefit. Experiment B has a much higher accuracy than Experiment A (0.969 compared with 0.834), and its std is smaller (0.02 compared with 0.038), so it is also more stable between folds.

**3. Why is Experiment B expected to perform better?**

The three identical trees always make the same predictions and the same mistakes, so the majority vote is always the same as one tree, and the other two votes add nothing. In Experiment B, the three algorithms learn in different ways (a linear model, a distance-based model, and a kernel model), so they make errors on different samples. When one model is wrong, the other two can outvote it, so the errors cancel out. This is the diversity that makes an ensemble work.

**4. What would happen if all base models achieved 99% accuracy individually? Can the ensemble beat this?**

The ensemble could only improve slightly, because there is almost no room left (the maximum is 100%). It can beat 99% if the models make their few mistakes on different samples, because then at least 2 of the 3 models are usually right and the vote fixes the single wrong one. But if all models are wrong on the same samples (low diversity), the vote is wrong too and the ensemble stays at about 99%. So the gain from an ensemble depends on diversity, and with very accurate models it is small.